# Quick LLM tests

Manual side by side comparison of the three models on a few questions. Gives an intuition before running the full benchmark in `02_experiments`.

In [1]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.generation import LLMGenerator
from src.rag_pipeline import RAGPipeline

In [2]:
# embedder + chromadb loaded once
emb = Embedder('OrdalieTech/Solon-embeddings-base-0.1')
vs = VectorStore('../data/chroma_db', 'collection_solon_base')
print('chunks:', vs.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunks: 1323


In [3]:
# test questions (one per source type, written in French because the corpus is French)
questions = [
    "Dois-je nommer un DPO si j'ai 30 salaries ?",
    "Combien de temps puis-je conserver un CV de candidat non retenu ?",
    "Pourquoi Google a ete sanctionne en 2019 ?",
]

## Loop over the three models

Smallest to largest. Memory is released between each to avoid saturation.

In [4]:
def free_mem():
    gc.collect()
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

models = [
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
]

llm = None
for m in models:
    print('=' * 60)
    print('model:', m)
    if llm is not None:
        del llm
    free_mem()
    llm = LLMGenerator(m)
    pipe = RAGPipeline(emb, vs, llm, prompt_name='citation')
    for q in questions:
        print()
        print('Q:', q)
        resp = pipe.answer(q)
        print('A:', resp.answer[:600])
        print('  total latency:', resp.latency_ms['total'], 'ms')

model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Q: Dois-je nommer un DPO si j'ai 30 salaries ?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Q: Dois-je nommer un DPO si j'ai 30 salaries ?
A: [Source: CNIL fiche dpo] "Est-il possible pour un salarié de refuser d'être désigné DPO ? Les règles générales du droit du travail s'appliquent ici : si cette désignation constitue une modification substantielle du contrat de travail, alors la personne doit être mise en position de la refuser."
  total latency: 12073.1 ms

Q: Combien de temps puis-je conserver un CV de candidat non retenu ?
A: Selon le contexte fourni, vous pouvez conserver le CV de candidat non retenu jusqu'à deux années à compter du dernier contact avec celui-ci. Cette durée peut être considérée comme une durée recommandée par la CNIL, mais elle n'est pas obligatoire selon le texte.
  total latency: 4921.4 ms

Q: Pourquoi Google a ete sanctionne en 2019 ?
A: La raison pour laquelle Google a été sanctionné en 2019 était principalement due à l'irrégularité de la procédure de contrôle effectué par la Commission Nationale de l'informatique et des libertés (CNIL). Selon l

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Q: Dois-je nommer un DPO si j'ai 30 salaries ?
A: Pour répondre à cette question, nous devons considérer plusieurs éléments du contexte fourni. Tout d'abord, la taille de l'entreprise n'est pas spécifiquement mentionnée comme critère direct pour la désignation d'un DPO. Cependant, le contexte indique que la désignation d'un DPO est requise "si elle est responsable de traitement" et non simplement en raison de la taille de l'entreprise. 

Ensuite, le contexte mentionne que la désignation d'un DPO peut être effectuée par un salarié de l'entreprise, ce qui suggère qu'il n'y a pas de limite en termes de nombre de salariés pour pouvoir désigner un
  total latency: 42932.5 ms

Q: Combien de temps puis-je conserver un CV de candidat non retenu ?
A: Selon les informations fournies, pour les candidats non retenus, les données figurant dans la base de données mise en œuvre dans le cadre d'une CV-thèque par des intermédiaires (ex : cabinets de recrutement) ou des employeurs, il est recommandé de

## Observations

- TinyLlama: often answers in english even when the question is french. translates "salaries" as wages, gets confused. very generic output, does not really ground on the retrieved context.
- Qwen 1.5B: answers in french at least. sometimes picks a side fact from the context instead of the main answer (DPO question). good on the CV question. confuses years on the Google one (mixes up 2019 with a 2025 ruling).
- Qwen 3B: best of the three. structured answers in french, cites sources when relevant. on the Google 2019 question it says the info is not in the context instead of making things up, which is honest. slower though, around 30 to 40s per answer on M3 Pro.

Conclusion: Qwen 3B is the one to keep for the demo. TinyLlama is good only as a baseline to show the scale / quality trade off in the slides.
